<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [يوري كاشنيتسكي](https://yorko.github.io). تمت الترجمة والتحرير بواسطة [كريستينا بوتسكو](https://www.linkedin.com/in/christinabutsko/)، و[نرسيس باجيان](https://www.linkedin.com/in/nersesbagiyan/)، و[يوليا كليموشينا](https://www.linkedin.com/in/yuliya-klimushina-7168a9139)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center> الموضوع 4. التصنيف الخطي والانحدار
## <center> الجزء 4. حيث يكون الانحدار اللوجستي جيدًا وأين لا يكون كذلك
    
            
## الخطوط العريضة للمادة
1. [تحليل مراجعات أفلام IMDB](#1.-تحليل مراجعات أفلام IMDB)
2. [عدد بسيط من الكلمات](#2.-أ-عدد بسيط من الكلمات)
3. [مشكلة-XOR](#3.-مشكلة-XOR)
4. [مهمة تجريبية](#4.-مهمة تجريبية)
5. [موارد مفيدة](#5.-موارد-مفيدة)



## 1. تحليل مراجعات الأفلام على موقع IMDB



الآن القليل من الممارسة! نريد حل مشكلة التصنيف الثنائي لمراجعات أفلام IMDB. لدينا مجموعة تدريب تحتوي على تقييمات مميزة، 12500 تقييمًا تم تصنيفها على أنها جيدة، و12500 تقييمًا آخر سيئًا. هنا، ليس من السهل البدء بالتعلم الآلي على الفور لأنه ليس لدينا المصفوفة $X$؛ نحن بحاجة لإعداده. سوف نستخدم أسلوبًا بسيطًا: نموذج حقيبة الكلمات. سيتم تمثيل ميزات المراجعة من خلال مؤشرات وجود كل كلمة من المجموعة الكاملة في هذه المراجعة. المجموعة هي مجموعة جميع مراجعات المستخدمين. الفكرة موضحة بالصورة
<img src="../../img/bag_of_words.svg" width=80% />


In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_files
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

**للبدء، نقوم تلقائيًا بتنزيل مجموعة البيانات من [هنا](http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz) وإلغاء أرشفتها مع بقية مجموعات البيانات في مجلد البيانات. تم وصف مجموعة البيانات بإيجاز [هنا](http://ai.stanford.edu/~amaas/data/sentiment/). هناك 12.5 ألفًا من التقييمات الجيدة والسيئة في مجموعات الاختبار والتدريب.**


In [ ]:
import tarfile
from io import BytesIO

import requests

url = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"


def load_imdb_dataset(extract_path="../../data", overwrite=False):
    # check if existed already
    if (
        os.path.isfile(os.path.join(extract_path, "aclImdb", "README"))
        and not overwrite
    ):
        print("IMDB dataset is already in place.")
        return

    print("Downloading the dataset from:  ", url)
    response = requests.get(url)

    tar = tarfile.open(mode="r:gz", fileobj=BytesIO(response.content))

    data = tar.extractall(extract_path)


load_imdb_dataset()

In [ ]:
# change if you have it in alternative location
PATH_TO_IMDB = "../../data/aclImdb"

reviews_train = load_files(
    os.path.join(PATH_TO_IMDB, "train"), categories=["pos", "neg"]
)
text_train, y_train = reviews_train.data, reviews_train.target

reviews_test = load_files(os.path.join(PATH_TO_IMDB, "test"), categories=["pos", "neg"])
text_test, y_test = reviews_test.data, reviews_test.target

In [ ]:
# # Alternatively, load data from previously pickled objects.
# import pickle
# with open('../../data/imdb_text_train.pkl', 'rb') as f:
#     text_train = pickle.load(f)
# with open('../../data/imdb_text_test.pkl', 'rb') as f:
#     text_test = pickle.load(f)
# with open('../../data/imdb_target_train.pkl', 'rb') as f:
#     y_train = pickle.load(f)
# with open('../../data/imdb_target_test.pkl', 'rb') as f:
#     y_test = pickle.load(f)

In [ ]:
print("Number of documents in training data: %d" % len(text_train))
print(np.bincount(y_train))
print("Number of documents in test data: %d" % len(text_test))
print(np.bincount(y_test))


**إليك بعض الأمثلة على المراجعات.**


In [ ]:
print(text_train[1])

In [ ]:
y_train[1]  # bad review

In [ ]:
text_train[2]

In [ ]:
y_train[2]  # good review

In [ ]:
# import pickle
# with open('../../data/imdb_text_train.pkl', 'wb') as f:
#     pickle.dump(text_train, f)
# with open('../../data/imdb_text_test.pkl', 'wb') as f:
#     pickle.dump(text_test, f)
# with open('../../data/imdb_target_train.pkl', 'wb') as f:
#     pickle.dump(y_train, f)
# with open('../../data/imdb_target_test.pkl', 'wb') as f:
#     pickle.dump(y_test, f)


## 2. عدد بسيط من الكلمات



**أولاً، سنقوم بإنشاء قاموس لجميع الكلمات باستخدام CountVectorizer**


In [ ]:
cv = CountVectorizer()
cv.fit(text_train)

len(cv.vocabulary_)


**إذا نظرت إلى أمثلة "الكلمات" (دعنا نسميها الرموز المميزة)، يمكنك أن ترى أننا أهملنا العديد من الخطوات المهمة في معالجة النص (يمكن أن تكون المعالجة التلقائية للنص في حد ذاتها سلسلة منفصلة تمامًا من المقالات).**


In [ ]:
print(cv.get_feature_names()[:50])
print(cv.get_feature_names()[50000:50050])


**ثانيًا، نقوم بتشفير الجمل من نصوص المجموعة التدريبية مع فهارس الكلمات الواردة. سنستخدم التنسيق المتناثر.**


In [ ]:
X_train = cv.transform(text_train)
X_train


**دعونا نرى كيف نجح التحول لدينا**


In [ ]:
print(text_train[19726])

In [ ]:
X_train[19726].nonzero()[1]

In [ ]:
X_train[19726].nonzero()


**ثالثًا، سنطبق نفس العمليات على مجموعة الاختبار**


In [ ]:
X_test = cv.transform(text_test)


**الخطوة التالية هي تدريب الانحدار اللوجستي.**


In [ ]:
%%time
logit = LogisticRegression(solver="lbfgs", n_jobs=-1, random_state=7)
logit.fit(X_train, y_train)


**دعونا نلقي نظرة على الدقة في كل من مجموعات التدريب والاختبار.**


In [ ]:
round(logit.score(X_train, y_train), 3), round(logit.score(X_test, y_test), 3),


**يمكن عرض معاملات النموذج بشكل جميل.**


In [ ]:
def visualize_coefficients(classifier, feature_names, n_top_features=25):
    # get coefficients with large absolute values
    coef = classifier.coef_.ravel()
    positive_coefficients = np.argsort(coef)[-n_top_features:]
    negative_coefficients = np.argsort(coef)[:n_top_features]
    interesting_coefficients = np.hstack([negative_coefficients, positive_coefficients])
    # plot them
    plt.figure(figsize=(15, 5))
    colors = ["red" if c < 0 else "blue" for c in coef[interesting_coefficients]]
    plt.bar(np.arange(2 * n_top_features), coef[interesting_coefficients], color=colors)
    feature_names = np.array(feature_names)
    plt.xticks(
        np.arange(1, 1 + 2 * n_top_features),
        feature_names[interesting_coefficients],
        rotation=60,
        ha="right",
    );

In [ ]:
def plot_grid_scores(grid, param_name):
    plt.plot(
        grid.param_grid[param_name],
        grid.cv_results_["mean_train_score"],
        color="green",
        label="train",
    )
    plt.plot(
        grid.param_grid[param_name],
        grid.cv_results_["mean_test_score"],
        color="red",
        label="test",
    )
    plt.legend();

In [ ]:
visualize_coefficients(logit, cv.get_feature_names())


**لتحسين نموذجنا، يمكننا تحسين معامل التنظيم لـ `Logistic Regression`. سنستخدم `sklearn.pipeline` لأنه يجب تطبيق `CountVectorizer` فقط على بيانات التدريب (حتى لا "نلقي نظرة خاطفة" على مجموعة الاختبار ولا نحسب ترددات الكلمات هناك). في هذه الحالة، `pipeline` يحدد التسلسل الصحيح للإجراءات: قم بتطبيق `CountVectorizer`، ثم قم بالتدريب `Logistic Regression`.**


In [ ]:
%%time
from sklearn.pipeline import make_pipeline

text_pipe_logit = make_pipeline(
    CountVectorizer(),
    # for some reason n_jobs > 1 won't work
    # with GridSearchCV's n_jobs > 1
    LogisticRegression(solver="lbfgs", n_jobs=1, random_state=7),
)

text_pipe_logit.fit(text_train, y_train)
print(text_pipe_logit.score(text_test, y_test))

In [ ]:
%%time
from sklearn.model_selection import GridSearchCV

param_grid_logit = {"logisticregression__C": np.logspace(-5, 0, 6)}
grid_logit = GridSearchCV(
    text_pipe_logit, param_grid_logit, return_train_score=True, cv=3, n_jobs=-1
)

grid_logit.fit(text_train, y_train)


**دعونا نطبع أفضل $C$ ونتائج السيرة الذاتية باستخدام هذه المعلمة الفائقة:**


In [ ]:
grid_logit.best_params_, grid_logit.best_score_

In [ ]:
plot_grid_scores(grid_logit, "logisticregression__C")


لمجموعة التحقق من الصحة:


In [ ]:
grid_logit.score(text_test, y_test)

**الآن لنفعل الشيء نفسه مع الغابة العشوائية. ونحن نرى أنه من خلال الانحدار اللوجستي، نحقق دقة أفضل بجهد أقل.**


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
forest = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=17)

In [ ]:
%%time
forest.fit(X_train, y_train)

In [ ]:
round(forest.score(X_test, y_test), 3)


### 3. مشكلة XOR
لنفكر الآن في مثال تكون فيه النماذج الخطية أسوأ.
لا تزال طرق التصنيف الخطي تحدد سطحًا فاصلًا بسيطًا للغاية - السطح الزائد. أشهر مثال على الألعاب حيث لا يمكن تقسيم الفئات بواسطة مستوى مفرط (أو خط) بدون أخطاء هو "مشكلة XOR".
XOR هي دالة "OR الحصرية"، وهي دالة منطقية تحتوي على جدول الحقيقة التالي:
<img src='../../img/XOR_table.gif'>
XOR هو الاسم الذي يطلق على مشكلة التصنيف الثنائي البسيطة التي يتم فيها تقديم الفئات كسحب نقطية متقاطعة ممتدة قطريًا.


In [ ]:
# creating dataset
rng = np.random.RandomState(0)
X = rng.randn(200, 2)
y = np.logical_xor(X[:, 0] > 0, X[:, 1] > 0)

In [ ]:
plt.scatter(X[:, 0], X[:, 1], s=30, c=y, cmap=plt.cm.Paired);


ومن الواضح أنه لا يمكن رسم خط مستقيم واحد للفصل بين فئة وأخرى دون أخطاء. ولذلك، أداء الانحدار اللوجستي ضعيف مع هذه المهمة.


In [ ]:
def plot_boundary(clf, X, y, plot_title):
    xx, yy = np.meshgrid(np.linspace(-3, 3, 50), np.linspace(-3, 3, 50))
    clf.fit(X, y)
    # plot the decision function for each datapoint on the grid
    Z = clf.predict_proba(np.vstack((xx.ravel(), yy.ravel())).T)[:, 1]
    Z = Z.reshape(xx.shape)

    image = plt.imshow(
        Z,
        interpolation="nearest",
        extent=(xx.min(), xx.max(), yy.min(), yy.max()),
        aspect="auto",
        origin="lower",
        cmap=plt.cm.PuOr_r,
    )
    contours = plt.contour(xx, yy, Z, levels=[0], linewidths=2, linetypes="--")
    plt.scatter(X[:, 0], X[:, 1], s=30, c=y, cmap=plt.cm.Paired)
    plt.xticks(())
    plt.yticks(())
    plt.xlabel(r"$x_1$")
    plt.ylabel(r"$x_2$")
    plt.axis([-3, 3, -3, 3])
    plt.colorbar(image)
    plt.title(plot_title, fontsize=12);

In [ ]:
plot_boundary(
    LogisticRegression(solver="lbfgs"), X, y, "Logistic Regression, XOR problem"
)


ولكن إذا تم إعطاء ميزات متعددة الحدود كمدخل (هنا، حتى درجتين)، فسيتم حل المشكلة.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
logit_pipe = Pipeline(
    [
        ("poly", PolynomialFeatures(degree=2)),
        ("logit", LogisticRegression(solver="lbfgs")),
    ]
)

In [ ]:
plot_boundary(logit_pipe, X, y, "Logistic Regression + quadratic features. XOR problem")


هنا، لا يزال الانحدار اللوجستي ينتج مستوى فائقًا ولكن في مساحة ميزات سداسية الأبعاد $1, x_1, x_2, x_1^2, x_1x_2$ و$x_2^2$. عندما نسقط على مساحة الميزة الأصلية، $x_1, x_2$، تكون الحدود غير خطية.
من الناحية العملية، تساعد الميزات متعددة الحدود، ولكن بناءها بشكل صريح غير فعال من الناحية الحسابية. يعمل SVM مع خدعة kernel بشكل أسرع بكثير. في هذا النهج، يتم فقط حساب المسافة بين الكائنات (المحددة بواسطة وظيفة kernel) في مساحة عالية الأبعاد، وليست هناك حاجة لإنتاج عدد كبير من الميزات بشكل اندماجي. 


## 4. مهمة تجريبية
للتدرب على النماذج الخطية، يمكنك إكمال [هذه المهمة](https://www.kaggle.com/kashnitsky/a4-demo-sarcasm-detection-with-logit) حيث ستقوم ببناء نموذج للكشف عن السخرية. هذه المهمة مخصصة لك فقط للتدرب عليها، وهي تتوافق مع [الحل](https://www.kaggle.com/kashnitsky/a4-demo-sarcasm-detection-with-logit-solution).
## 5. موارد مفيدة
- متوسط ["قصة"](https://medium.com/open-machine-learning-course/open-machine-learning-course-topic-4-linear-classification-and-regression-44a41b9b5220) استنادًا إلى دفتر الملاحظات هذا
- الطبق الرئيسي [الموقع](https://mlcourse.ai)، [مستودع الدورة](https://github.com/Yorko/mlcourse.ai)، ويوتيوب [القناة](https://www.youtube.com/watch?v=QKTuw4PNOsU&list=PLVlY_7IJCMJeRfZ68eVfEcu-UcN9BbwiX)
- مواد الدورة التدريبية باعتبارها [مجموعة بيانات Kaggle](https://www.kaggle.com/kashnitsky/mlcourse)
- إذا كنت تقرأ اللغة الروسية: [مقال](https://habrahabr.ru/company/ods/blog/323890/) على موقع Habr.com يحتوي على نفس المادة. و[محاضرة](https://youtu.be/oTXGQ-_oqvI) على اليوتيوب
- نظرة عامة لطيفة وموجزة على النماذج الخطية مقدمة في كتاب ["التعلم العميق"](http://www.deeplearningbook.org) (I. Goodfellow، Y. Bengio، و A. Courville).
- تتم تغطية النماذج الخطية عمليا في كل كتاب تعلم الآلة. نوصي بـ "التعرف على الأنماط والتعلم الآلي" (C. Bishop) و"التعلم الآلي: منظور احتمالي" (K. Murphy).
- إذا كنت تفضل نظرة شاملة على النموذج الخطي من وجهة نظر الإحصائي، فاطلع على "عناصر التعلم الإحصائي" (T. Hastie، R. Tibshirani، و J. Friedman).
- سيرشدك كتاب "التعلم الآلي أثناء العمل" (P. Harrington) عبر تطبيقات خوارزميات تعلم الآلة الكلاسيكية في لغة بايثون النقية.
- مكتبة [Scikit-learn](http://scikit-learn.org/stable/documentation.html). هؤلاء الرجال يعملون بجد لكتابة وثائق واضحة حقًا.
- Scipy 2017 [برنامج تعليمي لـ scikit-learn] (https://github.com/amueller/scipy-2017-sklearn) بواسطة Alex Gramfort وAndreas Mueller.
- [دورة تعلم الآلة] (https://github.com/diefimov/MTH594_MachineLearning) إضافية بمواد جيدة جدًا.
- [تطبيقات](https://github.com/rushter/MLAlgorithms) للعديد من خوارزميات تعلم الآلة. البحث عن الانحدار الخطي والانحدار اللوجستي.